# SpectFbCalc Tutorial

## This notebook demonstrates how to calculate broadband and spectral climate feedbacks using the provided sample data.

To use the notebook download kernels: radiative kernels (spectral and broadband) are hosted separately on Zenodo
and Mendeley Data and are not included in the repository. 
Download them with:
```bash
cd spectfbcalc/
nohup python download_data.py > download_data.log 2>&1 &
disown
```
This runs in the background. Track progress with:
```bash
tail -f download_data.log
```
If interrupted, simply re-run the same command: already-downloaded files are skipped automatically.

## Import libraries and modules

In [ ]:
import spectfbcalc_lib as sfc
from climtools import climtools_lib as ctl 
import output_lib as out

In [ ]:
# test libraries import
sfc.mytestfunction()

In [ ]:
ctl.datestamp()

In [4]:
import sys
import os
import glob

import numpy as np
import xarray as xr

from matplotlib import pyplot as plt
import matplotlib.cbook as cbook

### OPTIONAL: launch workers to speed up the process (needs a SLURM scheduler)

In [ ]:
from dask_jobqueue import SLURMCluster
from dask.distributed import Client

# dask will automatically submit SLURM jobs for you
cluster = SLURMCluster(
    cores=4,
    memory="64GB",
    processes=4,
    walltime="01:00:00",
    job_extra_directives=[
        "--account=spitfabi",
        "--qos=np"
    ]
)

# dask scale to desired number of workers
cluster.scale(jobs=4)  # This submits 4 SLURM jobs

# connect client
client = Client(cluster)

In [ ]:
print(client.dashboard_link)

In [ ]:
print(client)

In [ ]:
import dask.array as da
x = da.random.random((20000, 20000), chunks=(1000, 1000))
result = (x + x.T).mean().compute()
print(result)

# to check the status of the workers and the number of tasks executed, you can use the following code:
info = client.scheduler_info()['workers']
for addr, w in info.items():
    print(addr, "- tasks:", w.get('metrics', {}).get('task_counts', 'n/a'))

## Load configuration

In [ ]:
config_file='config_template.yaml'
config = sfc.load_config(config_file, variable_mapping_file = None)
ref_dir = config['file_paths']['reference_dataset']
exp_dir = config['file_paths']['experiment_dataset']
cart_out_huang = os.path.join(config['file_paths']['output'], 'HUANG/')
cart_out_spect = os.path.join(config['file_paths']['output'], 'SPECTRAL/')
os.makedirs(cart_out_huang, exist_ok=True)
os.makedirs(cart_out_spect, exist_ok=True)

## Broadband feedbacks (HUANG kernels)

### Tutorial Step-by-Step
### Load the control experiment object directly, data are already preprocessed and remapped.

In [ ]:
ker = 'HUANG'  
raw_variables = {"hus", "rlut", "rsdt", "rlutcs", "rsus", "rsds", "rsut", "rsutcs", "ta", "tas", "ts"}

control_h = sfc.Experiment('PI', orig_dir='', remap_dir=ref_dir, raw_variables=raw_variables, variable_mapping=config['variable_mapping'], file_dict={})
experiment_h = sfc.Experiment('4x', orig_dir='', remap_dir=exp_dir, raw_variables=raw_variables, variable_mapping=config['variable_mapping'], file_dict={})
control_h.load_remapped()
experiment_h.load_remapped()

### Inizialized kernel object

In [ ]:
kernel = sfc.Kernel(ker, config=config)
k = kernel.kernel[('clr', 't')]

control_h.check_coords()
control_h.vertical_interp(k)
experiment_h.check_coords()
experiment_h.vertical_interp(k)

### Compute climate anomalies (dT) and radiative anomalies (dRt)

In [ ]:
# compute albedo
control_h.check_albedo()
experiment_h.check_albedo()

# check water vapor
control_h.check_vars('water-vapor', kernel.wv_name)
experiment_h    .check_vars('water-vapor', kernel.wv_name)

# compute net TOA
control_h.compute_net_TOA()
experiment_h.compute_net_TOA()

In [ ]:
# compute climate anomalies (4x - PI)
sfc.compute_anomalies(experiment_h, control_h, method=config['anomaly_method'], time_range_clim=config['time_range_clim'])

In [ ]:
# compute radiative anomalies one by one
sfc.Rad_anomaly_planck_surf(experiment_h, kernel, cart_out_huang, save_pattern=True)

In [ ]:
sfc.Rad_anomaly_albedo(experiment_h, kernel, cart_out_huang, save_pattern=True)

In [ ]:
sfc.Rad_anomaly_planck_atm_lr(experiment_h, kernel, cart_out_huang, save_pattern=True)

In [ ]:
sfc.Rad_anomaly_wv(experiment_h, control_h, kernel, cart_out_huang, save_pattern=True)

In [ ]:
sfc.Rad_anomaly_cloud(experiment_h, cart_out_huang, save_pattern=True)

### Compute feedbacks

In [ ]:
# compute all feedbacks 
fb_huang = sfc.calc_fb_from_exp(experiment_h, control_h, kernel, cart_out_huang, save_pattern=config['save_pattern'], num_year_fb =config['num_year_regr'])

In [ ]:
# compute interannual feedbacks 
fb_huang_interannual = sfc.calc_fb_interannual(experiment_h, control_h, kernel, cart_out_huang)

## Spectral feedbacks 

To compute spectral feedbacks, the tool needs to remap the data to the high-resolution spectral grid. We can use the wrapper `preprocess_data` to automate loading, remapping, and anomaly computation in a single step.

### Preprocess data

In [ ]:
control_sp, experiment_sp, kernel_sp = sfc.preprocess_data(config=config, ker='SPECTRAL')

### Calculate spectral feedbacks 

In [ ]:
fb_spectral = sfc.calc_fb_spectral(
    experiment=experiment_sp, 
    kernel=kernel_sp, 
    cart_out=cart_out_spect,
    control=control_sp,
    save_pattern=config['save_pattern'],
    num_year_fb=config['num_year_regr']
)

## Save output and plot example

### Broadband

In [ ]:
import xarray as xr

out_dir = 'path/to/output/directory/'  # Replace with your desired output directory

out_txt_huang = out_dir + 'res_fb_huang.txt'
out_nc_huang = out_dir + 'res_fb_patterns_huang.nc'

# Salva
out.save_feedback_output(fb_huang, out_path_txt=out_txt_huang, out_path_nc=out_nc_huang)

# Gregory plot
out.plot_feedback_slope(out_txt_huang, sim_label="Broadband (Huang) - Sample Data", save_path=out_dir+"gregory_huang.png")

# Spatial maps of feedback patterns
ds_patterns_huang = xr.open_dataset(out_nc_huang)
out.save_all_fb_patterns_to_pdf(
    ds=ds_patterns_huang, 
    output_folder=out_dir, 
    pdf_name="spatial_maps_huang.pdf",
    run_label="Broadband" 
)

# Radiative Budget Closure
print("Plotting Radiative Budget Closure (Broadband only)...")

dRt_dict = sfc.open_dRt(cart_out_huang, names=sfc.dRt_all + sfc.dRt_all_cloud)

# Clear sky
out.plot_toa_anomaly(
    experiment=experiment_h,  
    dRt_dict=dRt_dict, 
    title="Radiative Budget Closure (HUANG) - Clear Sky", 
    sky="clr", 
    output_file=out_dir + "budget_closure_clr.png"
)

# All sky
out.plot_toa_anomaly(
    experiment=experiment_h, 
    dRt_dict=dRt_dict, 
    title="Radiative Budget Closure (HUANG) - All Sky", 
    sky="cld", 
    output_file=out_dir + "budget_closure_cld.png"
)

### Spectral 

In [ ]:
out_txt_spect = out_dir + 'res_fb_spectral.txt'
out_nc_spect = out_dir + 'res_fb_patterns_spectral.nc'

out.save_feedback_output(fb_spectral, out_path_txt=out_txt_spect, out_path_nc=out_nc_spect)

out.plot_feedback_slope(out_txt_spect, sim_label="Spectral LW - Sample Data", save_path=out_dir+"gregory_spectral.png")

ds_patterns_spect = xr.open_dataset(out_nc_spect)
out.save_all_fb_patterns_to_pdf(
    ds=ds_patterns_spect, 
    output_folder=out_dir, 
    pdf_name="spatial_maps_spectral.pdf",
    run_label="Spectral",
    components=["planck-surf", "planck-atmo", "lapse-rate", "water-vapor-lw"] 
)